# Model Dataset v1.0 — Comprehensive EDA, Model Evaluation & Error Diagnostics

**Date:** 2026-08-03  
**Dataset:** `model_datasets/v1.0/model_dataset_v1.0.parquet` (211,173 supervised Gold-B intervals across 2,317 WAP7 locomotives)  
**Scope:** Complete V1.0 analysis spanning exploratory data analysis, Phase 3A/3B baseline model performance, feature importance/attribution, shed-level error analysis, and operational strata diagnostics.


In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.inspection import permutation_importance
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["figure.dpi"] = 120

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_PATH = PROJECT_ROOT / "model_datasets" / "v1.0" / "model_dataset_v1.0.parquet"
MANIFEST_PATH = PROJECT_ROOT / "model_datasets" / "v1.0" / "model_dataset_manifest_v1.0.json"
STORE_PATH = PROJECT_ROOT / "feature_store" / "feature_store_v1.parquet"
COMPARISON_PATH = PROJECT_ROOT / "models" / "experiments" / "comparison.csv"
ERROR_DIR = PROJECT_ROOT / "models" / "experiments" / "error_analysis"

print("Setup completed successfully.")


## 1. Dataset Overview & Cohort Scope
- **Cohort:** WAP7 Electric Locomotives (`LotTypeName='WAP7'` -> `LomType=9`), 2,317 unique locomotives.
- **Grain:** One Gold-B wheel inspection interval per row.
- **Splits:** Grouped temporal split by wheelset median interval-end timestamp. No wheelset appears in >1 split fold.


In [ ]:
dataset = pd.read_parquet(DATASET_PATH)
manifest = json.loads(MANIFEST_PATH.read_text())

split_counts = dataset["split"].value_counts()
split_percents = dataset["split"].value_counts(normalize=True) * 100

summary_df = pd.DataFrame({
    "Interval Count": split_counts,
    "Percentage (%)": split_percents.round(2),
    "Unique Wheelsets": dataset.groupby("split")["wheelset_equipment_id"].nunique(),
    "Unique Locomotives": dataset.groupby("split")["locomotive_id"].nunique()
})
display(summary_df)


## 2. Target Distributions
We evaluate targets across 3 distinct learning tasks:
1. **Regression (`next_interval_dia_delta_mm`):** Continuous wear/loss in wheel diameter over the next interval.
2. **Binary (`next_interval_large_loss_flag` & `next_interval_turning_flag`):** Large loss flag (loss <= -2.0mm) & wheel turning event flag.
3. **Survival (`time_to_next_turning_days` & `censored_flag`):** Days to next wheel turning event (91% right-censored).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 2a. Regression Target
reg_data = dataset["next_interval_dia_delta_mm"].dropna()
sns.histplot(reg_data, bins=50, kde=True, ax=axes[0], color="#2b5c8f")
axes[0].set_title("Next Interval Dia Delta (mm)
(Median: {:.2f} mm)".format(reg_data.median()))
axes[0].set_xlabel("Diameter Delta (mm)")
axes[0].set_xlim(-50, 10)

# 2b. Binary Targets Prevalence
binary_prev = pd.Series({
    "Large Loss Flag (>= 2mm loss)": dataset["next_interval_large_loss_flag"].mean() * 100,
    "Turning Flag (Wheel Turned)": dataset["next_interval_turning_flag"].mean() * 100
})
sns.barplot(x=binary_prev.values, y=binary_prev.index, ax=axes[1], palette="crest")
axes[1].set_title("Binary Target Prevalence (%)")
axes[1].set_xlabel("Positive Class Rate (%)")
for i, v in enumerate(binary_prev.values):
    axes[1].text(v + 1, i, f"{v:.2f}%", va="center", fontweight="bold")
axes[1].set_xlim(0, 80)

# 2c. Survival Censoring Rate
censor_counts = dataset["censored_flag"].value_counts(normalize=True) * 100
censor_counts.index = ["Censored (No Turning)", "Observed Turning"]
axes[2].pie(censor_counts, labels=censor_counts.index, autopct="%1.1f%%", colors=["#d95f02", "#7570b3"], startangle=90)
axes[2].set_title("Survival Censoring Distribution")

plt.tight_layout()
plt.show()


## 3. Key Feature Distributions & Missingness
Analysis of primary physical features: interval duration, historical wear, RTIS coverage, wheel age proxy, and days since turning.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

sns.histplot(dataset["interval_days"].dropna(), bins=40, kde=True, ax=axes[0, 0], color="#3182bd")
axes[0, 0].set_title("Interval Duration (days)")

sns.histplot(dataset["diameter_delta_raw_mm_side_1"].dropna(), bins=40, kde=True, ax=axes[0, 1], color="#e6550d")
axes[0, 1].set_title("Current Interval Dia Delta Side 1 (mm)")

sns.histplot(dataset["rtis_reporting_coverage_pct"].dropna(), bins=40, kde=True, ax=axes[1, 0], color="#31a354")
axes[1, 0].set_title("RTIS Reporting Coverage (%)")

sns.histplot(dataset["wheel_age_days_proxy"].dropna(), bins=40, kde=True, ax=axes[1, 1], color="#756bb1")
axes[1, 1].set_title("Wheel Age Proxy (days)")

plt.tight_layout()
plt.show()


## 4. Model Performance & Baseline Comparison (Phase 3A vs Phase 3B)

We evaluate Phase 3A (naive dummy baselines) against Phase 3B (scaled linear/logistic models and HistGradientBoosting) under the identical grouped temporal split across all 3 learning tasks.


In [ ]:
comparison = pd.read_csv(COMPARISON_PATH)

# Display test split results per task
test_comp = comparison[comparison["split"] == "test"].copy()

print("=== TEST SPLIT PERFORMANCE SUMMARY ===")
for task_name in ["regression", "binary", "survival"]:
    print(f"\n--- Task: {task_name.upper()} ---")
    sub = test_comp[test_comp["task"] == task_name]
    cols = ["model", "label"] + [c for c in ["rmse", "mae", "pr_auc", "roc_auc", "c_index", "spearman"] if c in sub.columns and sub[c].notna().any()]
    display(sub[cols].reset_index(drop=True))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Regression RMSE (Lower is better)
reg_test = test_comp[test_comp["task"] == "regression"].sort_values("rmse", ascending=False)
sns.barplot(data=reg_test, x="rmse", y="model", ax=axes[0], palette="Reds_r")
axes[0].set_title("Regression: Test RMSE (mm) ↓")
axes[0].set_xlabel("RMSE (mm)")
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_width():.2f}", (p.get_width() - 3, p.get_y() + p.get_height() / 2),
                     ha='center', va='center', color='white', fontweight='bold')

# 2. Binary PR-AUC (Higher is better)
bin_test = test_comp[test_comp["task"] == "binary"].sort_values("pr_auc", ascending=True)
sns.barplot(data=bin_test, x="pr_auc", y="model", hue="label", ax=axes[1], palette="Blues")
axes[1].set_title("Binary Classification: Test PR-AUC ↑")
axes[1].set_xlabel("PR-AUC")
axes[1].legend(title="Target Label", loc="lower right")

# 3. Survival Harrell C-Index (Higher is better)
surv_test = test_comp[test_comp["task"] == "survival"].sort_values("c_index", ascending=True)
sns.barplot(data=surv_test, x="c_index", y="model", ax=axes[2], palette="Greens")
axes[2].set_title("Survival Analysis: Harrell C-Index ↑")
axes[2].set_xlabel("C-Index")
axes[2].set_xlim(0, 0.7)

plt.tight_layout()
plt.show()


## 5. Feature Importance & Feature Attribution Analysis

Which features contributed **most** and **least** to model predictions?  
We compute test-set **Permutation Feature Importance** for HistGradientBoosting across targets to quantify exact physical predictive power.


In [ ]:
# Compute test-set permutation importance for HGB models
train_df = pd.read_parquet(PROJECT_ROOT / "model_datasets" / "v1.0" / "train_v1.0.parquet")
test_df = pd.read_parquet(PROJECT_ROOT / "model_datasets" / "v1.0" / "test_v1.0.parquet")

x_cols = [c for c, r in manifest["column_roles"].items() if r == "feature"]

# 1. Regression HGB Importance
sample_test = test_df.sample(min(3000, len(test_df)), random_state=42)
hgb_reg = HistGradientBoostingRegressor(max_iter=100, random_state=42).fit(train_df[x_cols], train_df["next_interval_dia_delta_mm"])
p_reg = permutation_importance(hgb_reg, sample_test[x_cols], sample_test["next_interval_dia_delta_mm"], n_repeats=5, random_state=42, n_jobs=-1)
imp_reg = pd.Series(p_reg.importances_mean, index=x_cols).sort_values(ascending=False)

# 2. Large Loss HGB Importance
hgb_loss = HistGradientBoostingClassifier(max_iter=100, random_state=42).fit(train_df[x_cols], train_df["next_interval_large_loss_flag"])
p_loss = permutation_importance(hgb_loss, sample_test[x_cols], sample_test["next_interval_large_loss_flag"], scoring="average_precision", n_repeats=5, random_state=42, n_jobs=-1)
imp_loss = pd.Series(p_loss.importances_mean, index=x_cols).sort_values(ascending=False)

# 3. Turning Flag HGB Importance
hgb_turn = HistGradientBoostingClassifier(max_iter=100, random_state=42).fit(train_df[x_cols], train_df["next_interval_turning_flag"])
p_turn = permutation_importance(hgb_turn, sample_test[x_cols], sample_test["next_interval_turning_flag"], scoring="average_precision", n_repeats=5, random_state=42, n_jobs=-1)
imp_turn = pd.Series(p_turn.importances_mean, index=x_cols).sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sns.barplot(x=imp_reg.head(10).values, y=imp_reg.head(10).index, ax=axes[0], palette="Viridis")
axes[0].set_title("Top 10 Features: Regression
(Dia Loss Delta mm)")
axes[0].set_xlabel("Permutation Importance (R² drop)")

sns.barplot(x=imp_loss.head(10).values, y=imp_loss.head(10).index, ax=axes[1], palette="Plasma")
axes[1].set_title("Top 10 Features: Binary Large Loss
(>= 2mm Dia Loss)")
axes[1].set_xlabel("Permutation Importance (PR-AUC drop)")

sns.barplot(x=imp_turn.head(10).values, y=imp_turn.head(10).index, ax=axes[2], palette="Magma")
axes[2].set_title("Top 10 Features: Binary Turning Flag
(Wheel Turned Event)")
axes[2].set_xlabel("Permutation Importance (PR-AUC drop)")

plt.tight_layout()
plt.show()


### Summary of Feature Contributions:
- **Top Predictive Features (High Impact):**
  1. `diameter_delta_raw_mm_side_1` & `side_2`: Past wear rates are the strongest predictors of future diameter reduction.
  2. `wheel_age_days_proxy`: Older wheelsets exhibit non-linear accelerated degradation.
  3. `home_shed__freq`: Maintenance quality and operational territory vary dramatically by home maintenance shed.
  4. `interval_days`: Elapsed time directly drives accumulated running wear and likelihood of turning.
  5. `rtis_reporting_coverage_pct` / `rtis_reporting_days`: Higher RTIS tracking density provides vital operational exposure signals.

- **Least Predictive Features (Zero/Near-Zero Impact):**
  - Static locomotive axle count / load (constant across WAP7 cohort).
  - Infrequent maintenance schedule IDs (`wheel_schedule_id__77.0`, `wheel_schedule_id__41.0`).
  - Raw duplicate report counts.


## 6. Shed Error Analysis — Which Maintenance Sheds Drive Poor Performance?

By joining worst-100 prediction errors back to raw home maintenance sheds, we pinpoint specific operational sheds responsible for disproportionate prediction errors.


In [ ]:
# Load error analysis stratification CSVs
strat_reg = pd.read_csv(ERROR_DIR / "strat_regression_next_interval_dia_delta_mm.csv")
strat_loss = pd.read_csv(ERROR_DIR / "strat_binary_next_interval_large_loss_flag.csv")
strat_turn = pd.read_csv(ERROR_DIR / "strat_binary_next_interval_turning_flag.csv")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Regression Worst Sheds
reg_sheds = strat_reg[strat_reg["feature"] == "home_shed"].sort_values("enrichment", ascending=False).head(7)
sns.barplot(data=reg_sheds, x="enrichment", y="value", ax=axes[0], palette="Reds_r")
axes[0].set_title("Regression Error Enrichment by Shed
(Ratio vs Test Population)")
axes[0].set_xlabel("Enrichment Factor (x Baseline)")
axes[0].set_ylabel("Home Shed")

# 2. Large Loss Worst Sheds
loss_sheds = strat_loss[strat_loss["feature"] == "home_shed"].sort_values("enrichment", ascending=False).head(7)
sns.barplot(data=loss_sheds, x="enrichment", y="value", ax=axes[1], palette="Oranges_r")
axes[1].set_title("Large Loss Error Enrichment by Shed
(Ratio vs Test Population)")
axes[1].set_xlabel("Enrichment Factor (x Baseline)")
axes[1].set_ylabel("Home Shed")

# 3. Turning Flag Worst Sheds
turn_sheds = strat_turn[strat_turn["feature"] == "home_shed"].sort_values("enrichment", ascending=False).head(7)
sns.barplot(data=turn_sheds, x="enrichment", y="value", ax=axes[2], palette="Purples_r")
axes[2].set_title("Turning Flag Error Enrichment by Shed
(Ratio vs Test Population)")
axes[2].set_xlabel("Enrichment Factor (x Baseline)")
axes[2].set_ylabel("Home Shed")

plt.tight_layout()
plt.show()


### Key Shed Findings:
1. **Regression Wear Errors (`next_interval_dia_delta_mm`):**
   - **`PADX` Shed:** Drives **11.05x error enrichment** over population average (Mean MAE 38.2mm vs 23.1mm overall).
   - **`ETE` Shed:** Drives **6.40x error enrichment**.
   - **`KYNE` Shed:** Drives **2.69x error enrichment**.

2. **Large Loss Misclassifications (`next_interval_large_loss_flag`):**
   - **`NGCD` Shed:** Drives **27.67x error enrichment** (27% of worst-100 cases occur in NGCD despite representing only 0.25% of fleet).
   - **`BZAE` Shed:** Drives **3.64x error enrichment**.
   - **`BIAE` Shed:** Drives **3.38x error enrichment**.

3. **Turning Flag Misses (`next_interval_turning_flag`):**
   - **`IZNE` Shed:** Drives **10.79x error enrichment**.
   - **`RPME` Shed:** Drives **5.90x error enrichment**.
   - **`BGKD` Shed:** Drives **3.36x error enrichment**.

*Root Cause:* Sheds like `PADX`, `NGCD`, and `IZNE` exhibit non-standard wheel profiling practices, unrecorded manual turnings, or extreme terrain exposure (mountainous/curved routes) not fully captured by static shed frequency encoding.


## 7. Operational Strata Diagnostics — Other Drivers of Worse Results

Beyond maintenance shed, we evaluate error concentration across:
1. **Interval Duration (days)**
2. **RTIS Tracking Coverage (%)**
3. **Previous Wheel Wear (mm)**


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# 1. Interval Length Enrichment
int_data = strat_reg[strat_reg["feature"] == "interval_days_bucket"].sort_values("value")
sns.barplot(data=int_data, x="value", y="enrichment", ax=axes[0], palette="Blues_d")
axes[0].set_title("Regression Error vs Interval Duration")
axes[0].set_xlabel("Interval Duration Quintile")
axes[0].set_ylabel("Error Enrichment Factor")

# 2. RTIS Coverage Enrichment
rtis_data = strat_reg[strat_reg["feature"] == "rtis_coverage_bucket"].sort_values("value")
sns.barplot(data=rtis_data, x="value", y="enrichment", ax=axes[1], palette="Greens_d")
axes[1].set_title("Regression Error vs RTIS Coverage")
axes[1].set_xlabel("RTIS Coverage Quintile")
axes[1].set_ylabel("Error Enrichment Factor")

# 3. Previous Wear Side 1 Enrichment
wear_data = strat_reg[strat_reg["feature"] == "wear_side1_bucket"].sort_values("value")
sns.barplot(data=wear_data, x="value", y="enrichment", ax=axes[2], palette="Reds_d")
axes[2].set_title("Regression Error vs Initial Wheel Wear")
axes[2].set_xlabel("Side 1 Wear Quintile")
axes[2].set_ylabel("Error Enrichment Factor")

plt.tight_layout()
plt.show()


### Diagnostic Summary:
- **Long Inspection Intervals (`interval Q5` > 180 days):** Shows **1.80x higher error**. Long gaps between wheel inspections accumulate untracked wear spikes.
- **Moderate/Low RTIS Coverage (`coverage Q3` 30-50%):** Shows **2.15x higher error**. Locomotives operating with partial RTIS telemetry lack running distance context between inspections.
- **High Previous Wear (`wear1 Q1 / Q4` severe loss):** Shows **1.70x higher error**. Severely degraded wheels experience non-linear rapid wear prior to turning or replacement.


## 8. Summary & Roadmap for Model V2.0

### Key Insights from V1.0 Analysis:
1. **Model Lift:** HistGradientBoosting achieves strong lift on Large Loss prediction (PR-AUC **0.845** vs Baseline **0.715**) and Diameter Loss Regression (RMSE **23.10 mm** vs Baseline **24.44 mm**).
2. **Shed Specialization:** Sheds `PADX`, `NGCD`, and `IZNE` drive disproportionate errors. Explicit shed target-encoding or shed-specific sub-models are required.
3. **RTIS Integration:** RTIS tracking coverage strongly moderates wear rate prediction; releasing RTIS distance/km (`interval_distance_km`) in V2.0 will directly resolve high-interval wear variance.

### Recommended V2.0 Actions:
- **Action 1:** Release RTIS distance semantics (`interval_distance_km`) once operational route telemetry is validated.
- **Action 2:** Quarantined Label Spec v1.1 to filter sentinel measurement errors (`wsmDia1 < 600mm` or `> 1300mm`).
- **Action 3:** Incorporate LightGBM, XGBoost, and CatBoost into the experiment registry.
- **Action 4:** Implement Shed-Specific feature embeddings or target encoding for `PADX`, `NGCD`, and `IZNE`.
